In [ ]:
# 1. Install txtify and LEANN
!pip install -q txtify leann





In [ ]:
#Do not use
# 2. Clean any partial clone and clone the official snap-stanford/memorilla repo
!rm -rf /content/memorilla
!git config --global http.version HTTP/1.1
!git clone https://github.com/snap-stanford/memorilla.git /content/memorilla

# Install Memorilla in editable mode
!pip install -q -e /content/memorilla

In [2]:
# 3. Download PDF directly from URL
import urllib.request

PDF_URL = "https://dairysciencepark.org/wp-content/uploads/2022/09/The-100-Ranking-of-Most-Influential-People-in-History-Michael-Hart-Citadel-Press-1992-PDFDrive-.pdf"  # <-- Replace with your PDF URL
PDF_FILENAME = "/content/LeanNInput.pdf"

req = urllib.request.Request(
    PDF_URL,
    headers={"User-Agent": "Mozilla/5.0"}
)
with urllib.request.urlopen(req) as response, open(PDF_FILENAME, "wb") as out_file:
    out_file.write(response.read())

print(f"Downloaded '{PDF_FILENAME}' from {PDF_URL}")




Downloaded '/content/LeanNInput.pdf' from https://dairysciencepark.org/wp-content/uploads/2022/09/The-100-Ranking-of-Most-Influential-People-in-History-Michael-Hart-Citadel-Press-1992-PDFDrive-.pdf


In [ ]:
import gc
import subprocess
from pathlib import Path
import torch
from leann import LeannBuilder, LeannSearcher, LeannChat

# --- Step 1: Extract text using txtify ---
PDF_FILENAME="/content/LeanNInput.pdf"
output_dir = Path("./txtify_out")
output_dir.mkdir(parents=True, exist_ok=True)

subprocess.run(["txtify", PDF_FILENAME, "-o", str(output_dir)], check=True)



In [ ]:
extracted_txt_path = "/content/txtify_out/LeanExtract.txt"

with open(extracted_txt_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print(f"[txtify] Extracted {len(lines)} lines.")

# --- Step 2: Insert every line into LEANN ---
INDEX_PATH = str(Path("./colab_doc.leann").resolve())
builder = LeannBuilder(backend_name="hnsw")

for idx, line in enumerate(lines, start=1):
    builder.add_text(line, metadata={"line_number": idx})

builder.build_index(INDEX_PATH)
print(f"[LEANN] Successfully indexed {len(lines)} lines.")


In [12]:




# --- Generate answer using lightweight Qwen3-0.6B ---
chat = LeannChat(
    INDEX_PATH,
    llm_config={"type": "hf", "model": "Qwen/Qwen3-0.6B"}
)

searcher = LeannSearcher(INDEX_PATH)

In [14]:
USER_QUERY = "What did Darwin do between 1831 and 1836?"
# --- Retrieve raw matching lines ---

leann_hits = searcher.search(USER_QUERY, top_k=3)
leann_response = chat.ask(USER_QUERY, top_k=3)



kwargs in HF:  {}


In [15]:
print(leann_response)

<think>

</think>

Between 1831 and 1836, Darwin set sail on the Beagle and returned home, during which he published several scientific works.


In [16]:
# --- Free GPU memory before loading Memorilla ---
del chat
del searcher
gc.collect()
torch.cuda.empty_cache()
print("[Memory] Cleared T4 VRAM for Memorilla.")

[Memory] Cleared T4 VRAM for Memorilla.


In [ ]:
# Do not Use
!pip install -q "git+https://github.com/snap-stanford/memorilla.git"

In [ ]:
#Do not use
from memorilla.vllm import MemoryVLLM

# Group lines into ~5-line blocks to prevent T4 OOM during embedding encoding
grouped_docs = [" ".join(lines[i:i+5]) for i in range(0, len(lines), 5)]

# Step 3: Load Memorilla with T4-safe memory flags
mem_model = MemoryVLLM.from_pretrained(
    "memorilla/Memorilla-Qwen3-8B",
    "stage3",
    memory_cache=False,      # Critical for 15GB T4 VRAM
    compile_memory=False,    # Avoids T4 Triton/compile OOM
    gpu_memory_utilization=0.85,
    max_model_len=2048
)

# Step 4: Query Memorilla directly with the document collection
memorilla_response = mem_model.answer(USER_QUERY, grouped_docs)




In [ ]:
#Do not Use
# Step 5: Share both responses side by side
print("\n" + "=" * 65)
print(f"QUERY: {USER_QUERY}")
print("=" * 65)

print("\n[LEANN] Top Retrieved Lines:")
for i, hit in enumerate(leann_hits, 1):
    print(f"  {i}. {hit}")

print(f"\n[LEANN] Answer:\n{leann_response}")
print("-" * 65)
print(f"\n[Memorilla] Answer:\n{memorilla_response}")
print("=" * 65)

In [ ]:
import base64
import os
import subprocess
import urllib.request
import zipfile

# Base64-encoded archive URL prevents browser/clipboard URL stripping
encoded_url = "aHR0cHM6Ly9naXRodWIuY29tL3NuYXAtc3RhbmZvcmQvbWVtb3JpbGxhL2FyY2hpdmUvcmVmcy9oZWFkcy9tYWluLnppcA=="
zip_url = base64.b64decode(encoded_url).decode("utf-8")
zip_path = "/content/memorilla.zip"

print("Fetching repository archive...")
req = urllib.request.Request(zip_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp, open(zip_path, "wb") as out_file:
    out_file.write(resp.read())

file_size = os.path.getsize(zip_path)
print(f"Downloaded {file_size:,} bytes.")

if not zipfile.is_zipfile(zip_path):
    with open(zip_path, "r", errors="ignore") as f:
        preview = f.read(300)
    raise RuntimeError(f"Download was blocked or redirected. Response preview:\n{preview}")

# Extract repository
subprocess.run(["rm", "-rf", "/content/memorilla", "/content/memorilla-main"], check=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall("/content")

os.rename("/content/memorilla-main", "/content/memorilla")
print("Extracted repository to /content/memorilla")



In [35]:
%%bash
# 1. Install Python 3.11 via Ubuntu apt (most reliable on Google Colab)
apt-get update -qq
apt-get install -y -qq software-properties-common
add-apt-repository -y ppa:deadsnakes/ppa
apt-get update -qq
apt-get install -y -qq python3.11 python3.11-venv python3.11-dev

# 2. Create the Python 3.11 virtual environment
rm -rf /content/py311_env
python3.11 -m venv /content/py311_env

# 3. Install Memorilla into the Python 3.11 environment using uv
uv pip install --python /content/py311_env/bin/python -e /content/memorilla

# 4. Verify the import
/content/py311_env/bin/python -c "from memorilla.vllm import MemoryVLLM; print('Success! Memorilla is ready.')"


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists...
Repository: 'Types: deb
URIs: https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/
Suites: noble
Components: main
'
Description:
This PPA contains more recent Python versions packaged for Ubuntu.

D

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  503  Service Unavailable [IP: 185.125.189.186 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
E: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/pool/main/p/python3.11/python3.11_3.11.18-1%2bnoble1_amd64.deb  503  Service Unavailable [IP: 185.125.189.186 443]
E: Fai

CalledProcessError: Command 'b'# 1. Install Python 3.11 via Ubuntu apt (most reliable on Google Colab)\napt-get update -qq\napt-get install -y -qq software-properties-common\nadd-apt-repository -y ppa:deadsnakes/ppa\napt-get update -qq\napt-get install -y -qq python3.11 python3.11-venv python3.11-dev\n\n# 2. Create the Python 3.11 virtual environment\nrm -rf /content/py311_env\npython3.11 -m venv /content/py311_env\n\n# 3. Install Memorilla into the Python 3.11 environment using uv\nuv pip install --python /content/py311_env/bin/python -e /content/memorilla\n\n# 4. Verify the import\n/content/py311_env/bin/python -c "from memorilla.vllm import MemoryVLLM; print(\'Success! Memorilla is ready.\')"\n'' returned non-zero exit status 127.